# Algorithmic Trading and Quantitative Strategies
## Part 5: Backtrader Deep Dive — Indicators, Signals, and Rules
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. Built-in Indicators
2. Custom Indicators
3. Signal Types and Crossover Detection
4. Order Types
5. Commission and Slippage
6. Sizers
7. Analyzers
8. Complete Example: SMA Crossover with Full Analysis
9. Exercises

## 1. Built-in Indicators

Backtrader provides a rich library of built-in technical indicators under `bt.indicators` (or `bt.ind` for short). When you define an indicator in `__init__`, it is automatically computed for every bar.

### Key Properties:
- Indicators are **lines** (time series) that are computed automatically
- They handle the minimum period requirement (e.g., SMA(200) won't be available until bar 200)
- They can be combined using arithmetic operations
- They can be used as inputs to other indicators

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import backtrader as bt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


# Download data for examples
spy_data = yf.download('SPY', start='2018-01-01', end='2024-12-31')
spy_data.columns = spy_data.columns.droplevel('Ticker')
print(f"Downloaded {len(spy_data)} bars for SPY")

### 1.1 Moving Averages

Moving averages smooth price data to identify the direction of the trend.

$$\text{SMA}(n) = \frac{1}{n} \sum_{i=0}^{n-1} P_{t-i}$$

$$\text{EMA}(n): E_t = \alpha \cdot P_t + (1-\alpha) \cdot E_{t-1}, \quad \alpha = \frac{2}{n+1}$$

$$\text{WMA}(n) = \frac{\sum_{i=0}^{n-1} (n-i) \cdot P_{t-i}}{\sum_{i=0}^{n-1} (n-i)}$$

In [ ]:
class ShowIndicators(bt.Strategy):
    """Strategy that demonstrates various built-in indicators."""
    
    def __init__(self):
        # Moving Averages
        self.sma20 = bt.indicators.SimpleMovingAverage(self.data.close, period=20)
        self.sma50 = bt.indicators.SimpleMovingAverage(self.data.close, period=50)
        self.ema20 = bt.indicators.ExponentialMovingAverage(self.data.close, period=20)
        self.wma20 = bt.indicators.WeightedMovingAverage(self.data.close, period=20)
        
    def next(self):
        pass  # Just plotting indicators

cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
cerebro.addstrategy(ShowIndicators)
cerebro.run()

cerebro.plot(iplot=False, volume=False, figsize=(16, 8))


### 1.2 RSI (Relative Strength Index)

RSI measures the speed and magnitude of recent price changes to evaluate overbought or oversold conditions.

$$\text{RSI} = 100 - \frac{100}{1 + RS}$$

where $RS = \frac{\text{Average Gain}}{\text{Average Loss}}$ over the lookback period.

- RSI > 70: Overbought (potential sell)
- RSI < 30: Oversold (potential buy)

In [ ]:
class RSIStrategy(bt.Strategy):
    params = dict(rsi_period=14, rsi_lower=30, rsi_upper=70, printlog=True)
    
    def __init__(self):
        self.rsi = bt.indicators.RSI(self.data.close, period=self.p.rsi_period)
        self.order = None
    
    def log(self, txt):
        if self.p.printlog:
            print(f"{self.data.datetime.date(0)}: {txt}")
    
    def notify_order(self, order):
        if order.status == order.Completed:
            if order.isbuy():
                self.log(f"BUY @ {order.executed.price:.2f}")
            else:
                self.log(f"SELL @ {order.executed.price:.2f}")
        self.order = None
    
    def notify_trade(self, trade):
        if trade.isclosed:
            self.log(f"TRADE CLOSED: PnL={trade.pnl:.2f}, Net={trade.pnlcomm:.2f}")
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            if self.rsi < self.p.rsi_lower:
                self.order = self.buy()
        else:
            if self.rsi > self.p.rsi_upper:
                self.order = self.close()

# Run RSI strategy
cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
cerebro.addstrategy(RSIStrategy, printlog=False)
cerebro.broker.setcash(100000)
cerebro.broker.setcommission(commission=0.001)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')

print(f"Starting: ${cerebro.broker.getvalue():,.2f}")
results = cerebro.run()
print(f"Final:    ${cerebro.broker.getvalue():,.2f}")

ta = results[0].analyzers.trades.get_analysis()
print(f"Total trades: {ta.total.total}")

### 1.3 MACD (Moving Average Convergence Divergence)

MACD is a trend-following momentum indicator:

$$\text{MACD Line} = \text{EMA}(12) - \text{EMA}(26)$$
$$\text{Signal Line} = \text{EMA}(9) \text{ of MACD Line}$$
$$\text{Histogram} = \text{MACD} - \text{Signal}$$

- Buy when MACD crosses above Signal Line
- Sell when MACD crosses below Signal Line

In [ ]:
class MACDStrategy(bt.Strategy):
    params = dict(
        fast=12, slow=26, signal=9,
        printlog=False,
    )
    
    def __init__(self):
        self.macd = bt.indicators.MACD(
            self.data.close,
            period_me1=self.p.fast,
            period_me2=self.p.slow,
            period_signal=self.p.signal,
        )
        self.crossover = bt.indicators.CrossOver(self.macd.macd, self.macd.signal)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()

cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
cerebro.addstrategy(MACDStrategy)
cerebro.broker.setcash(100000)
cerebro.broker.setcommission(commission=0.001)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)

print(f"Starting: ${cerebro.broker.getvalue():,.2f}")
results = cerebro.run()
print(f"Final:    ${cerebro.broker.getvalue():,.2f}")
cerebro.plot(iplot=False, volume=False, figsize=(16, 10))

### 1.4 Bollinger Bands

Bollinger Bands consist of a middle band (SMA) and upper/lower bands at a specified number of standard deviations.

$$\text{Middle} = \text{SMA}(n)$$
$$\text{Upper} = \text{SMA}(n) + k \cdot \sigma(n)$$
$$\text{Lower} = \text{SMA}(n) - k \cdot \sigma(n)$$

where $k$ is typically 2 and $\sigma(n)$ is the rolling standard deviation.

In [ ]:
class BollingerBandStrategy(bt.Strategy):
    params = dict(period=20, devfactor=2.0, printlog=False)
    
    def __init__(self):
        self.boll = bt.indicators.BollingerBands(
            self.data.close, period=self.p.period, devfactor=self.p.devfactor)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.data.close[0] < self.boll.lines.bot[0]:
                self.order = self.buy()
        else:
            if self.data.close[0] > self.boll.lines.top[0]:
                self.order = self.close()

cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
cerebro.addstrategy(BollingerBandStrategy)
cerebro.broker.setcash(100000)
cerebro.broker.setcommission(commission=0.001)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)

print(f"Starting: ${cerebro.broker.getvalue():,.2f}")
results = cerebro.run()
print(f"Final:    ${cerebro.broker.getvalue():,.2f}")
cerebro.plot(iplot=False, volume=False, figsize=(16, 10))

### 1.5 ATR (Average True Range)

ATR measures market volatility:

$$\text{TR} = \max(H-L, |H-C_{-1}|, |L-C_{-1}|)$$
$$\text{ATR}(n) = \text{SMA}(\text{TR}, n)$$

ATR is useful for:
- Setting stop-loss distances proportional to volatility
- Position sizing (volatility-adjusted)
- Identifying regime changes

In [ ]:
class ATRDemo(bt.Strategy):
    def __init__(self):
        self.atr = bt.indicators.ATR(self.data, period=14)
    
    def next(self):
        if len(self) >= len(self.data) - 3:
            print(f"{self.data.datetime.date(0)}: Close={self.data.close[0]:.2f}, "
                  f"ATR={self.atr[0]:.2f}, ATR%={self.atr[0]/self.data.close[0]*100:.2f}%")

cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
cerebro.addstrategy(ATRDemo)
cerebro.run()

## 2. Custom Indicators

You can create your own indicators by subclassing `bt.Indicator`.

In [ ]:
class TrendStrength(bt.Indicator):
    """
    Custom indicator: Trend Strength Index
    Measures the ratio of absolute directional movement to total movement.
    Values close to 1 indicate strong trends; close to 0 indicate choppy markets.
    """
    lines = ('tsi',)
    params = dict(period=20,)
    
    def __init__(self):
        abs_change = abs(self.data.close - self.data.close(-self.p.period))
        total_change = bt.indicators.SumN(abs(self.data.close - self.data.close(-1)), period=self.p.period)
        self.lines.tsi = abs_change / (total_change + 1e-10)


class CustomIndicatorDemo(bt.Strategy):
    def __init__(self):
        self.tsi = TrendStrength(self.data, period=20)
        self.sma = bt.indicators.SMA(self.data.close, period=50)
    
    def next(self):
        if len(self) >= len(self.data) - 3:
            print(f"{self.data.datetime.date(0)}: Close={self.data.close[0]:.2f}, "
                  f"TSI={self.tsi[0]:.3f}")

cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
cerebro.addstrategy(CustomIndicatorDemo)
cerebro.run()

## 3. Signal Types and Crossover Detection

Backtrader provides utility indicators for detecting crossovers:

- `bt.ind.CrossOver(fast, slow)`: Returns +1 when `fast` crosses above `slow`, -1 when below, 0 otherwise
- `bt.ind.CrossDown(fast, slow)`: Returns 1 when `fast` crosses below `slow`
- `bt.ind.CrossUp(fast, slow)`: Returns 1 when `fast` crosses above `slow`

In [ ]:
class CrossoverDemo(bt.Strategy):
    """Demonstrate different crossover types."""
    params = dict(fast=10, slow=30)
    
    def __init__(self):
        sma_fast = bt.ind.SMA(self.data.close, period=self.p.fast)
        sma_slow = bt.ind.SMA(self.data.close, period=self.p.slow)
        
        self.crossover = bt.ind.CrossOver(sma_fast, sma_slow)
        self.order = None
        self.trade_list = []
        self.entry_size = 0
    
    def notify_order(self, order):
        if order.status == order.Completed and order.isbuy():
            self.entry_size = order.executed.size
        self.order = None
    
    def notify_trade(self, trade):
        if trade.isclosed:
            entry_value = trade.price * abs(self.entry_size)
            self.trade_list.append({
                'entry_date': bt.num2date(trade.dtopen).date(),
                'exit_date': bt.num2date(trade.dtclose).date(),
                'pnl': trade.pnl,
                'pnl_pct': (trade.pnl / entry_value * 100) if entry_value != 0 else 0.0,
            })
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()
    
    def stop(self):
        if self.trade_list:
            df = pd.DataFrame(self.trade_list)
            print(f"\n{'='*60}")
            print(f"Trade Log (SMA {self.p.fast}/{self.p.slow} Crossover)")
            print(f"{'='*60}")
            print(df.to_string(index=False))
            print(f"\nTotal trades: {len(df)}")
            print(f"Win rate: {(df['pnl'] > 0).mean():.1%}")
            print(f"Avg PnL: ${df['pnl'].mean():.2f}")

cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
cerebro.addstrategy(CrossoverDemo)
cerebro.broker.setcash(100000)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
cerebro.run()

## 4. Order Types in Backtrader

Backtrader supports several order types:

| Type | Code | Description |
|:---|:---|:---|
| Market | `bt.Order.Market` | Fill at next bar's open |
| Close | `bt.Order.Close` | Fill at current bar's close |
| Limit | `bt.Order.Limit` | Fill when price reaches limit |
| Stop | `bt.Order.Stop` | Becomes market order when price hits stop |
| StopLimit | `bt.Order.StopLimit` | Becomes limit order when price hits stop |
| StopTrail | `bt.Order.StopTrail` | Trailing stop based on price or percentage |

In [ ]:
class LimitOrderDemo(bt.Strategy):
    """Demonstrate limit orders: buy 2% below current price."""
    
    def __init__(self):
        self.sma = bt.ind.SMA(self.data.close, period=20)
        self.order = None
        self.bar_count = 0
    
    def notify_order(self, order):
        if order.status == order.Completed:
            print(f"{self.data.datetime.date(0)}: {'BUY' if order.isbuy() else 'SELL'} "
                  f"@ {order.executed.price:.2f}")
        elif order.status in [order.Canceled, order.Expired]:
            print(f"{self.data.datetime.date(0)}: Order expired/cancelled")
        self.order = None
    
    def next(self):
        if self.order:
            return
        
        self.bar_count += 1
        if self.bar_count == 50 and not self.position:
            limit_price = self.data.close[0] * 0.98  # 2% below current
            self.order = self.buy(exectype=bt.Order.Limit, price=limit_price)
            print(f"{self.data.datetime.date(0)}: LIMIT BUY placed @ {limit_price:.2f} "
                  f"(current: {self.data.close[0]:.2f}), valid 10 bars")
        
        if self.position and self.bar_count > 100:
            self.order = self.close()

cerebro = bt.Cerebro()
data = yf.download('TUR', start='2020-01-01', end='2024-12-31')
data.columns = data.columns.droplevel('Ticker')
cerebro.adddata(bt.feeds.PandasData(dataname=data))
cerebro.addstrategy(LimitOrderDemo)
cerebro.broker.setcash(100000)
cerebro.run()

## 5. Commission and Slippage

Realistic backtesting requires modeling transaction costs.

### Commission Schemes

In [ ]:
# Different commission schemes
cerebro = bt.Cerebro()

# Percentage-based commission (most common for stocks)
cerebro.broker.setcommission(commission=0.001)  # 0.1% per trade

# Fixed commission per trade
# cerebro.broker.setcommission(commission=5.0, commtype=bt.CommInfoBase.COMM_FIXED)

# For futures (per-contract basis)
# cerebro.broker.setcommission(commission=2.0, margin=2000.0, mult=50.0)

# Slippage (fixed or percentage)
cerebro.broker.set_slippage_fixed(0.01)  # $0.01 per share
# cerebro.broker.set_slippage_perc(0.001)  # 0.1% slippage

print("Commission: 0.1% per trade")
print("Slippage: $0.01 per share (fixed)")

## 6. Sizers

Sizers determine **how many** shares/contracts to buy or sell. Instead of specifying size in every `buy()`/`sell()` call, you set a sizer once.

### Built-in Sizers
- `bt.sizers.FixedSize(stake=N)`: Always trade N shares
- `bt.sizers.PercentSizer(percents=P)`: Use P% of portfolio value
- `bt.sizers.AllInSizer(percents=P)`: Like PercentSizer but also sets for selling
- `bt.sizers.FixedReverser`: For bracket-style reversal trading

In [ ]:
class VolatilitySizer(bt.Sizer):
    """
    Custom sizer: position size inversely proportional to volatility.
    Risk a fixed fraction of capital per trade.
    """
    params = dict(
        risk_pct=0.02,     # Risk 2% of capital per trade
        atr_period=14,
        atr_multiplier=2,  # Stop loss at 2x ATR
    )
    
    def _getsizing(self, comminfo, cash, data, isbuy):
        atr = bt.indicators.ATR(data, period=self.p.atr_period)
        
        if len(atr) < self.p.atr_period:
            return 0
        
        current_atr = atr[0]
        if current_atr <= 0:
            return 0
        
        stop_distance = current_atr * self.p.atr_multiplier
        portfolio_value = self.broker.getvalue()
        risk_amount = portfolio_value * self.p.risk_pct
        
        size = int(risk_amount / stop_distance)
        return min(size, int(cash / data.close[0]))

# Demo: compare FixedSize vs PercentSizer
for sizer_name, sizer_class, sizer_kwargs in [
    ("FixedSize(100)", bt.sizers.FixedSize, dict(stake=100)),
    ("PercentSizer(95%)", bt.sizers.PercentSizer, dict(percents=95)),
]:
    cerebro = bt.Cerebro()
    cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
    cerebro.addstrategy(CrossoverDemo, fast=10, slow=30)
    cerebro.broker.setcash(100000)
    cerebro.broker.setcommission(commission=0.001)
    cerebro.addsizer(sizer_class, **sizer_kwargs)
    
    results = cerebro.run()
    print(f"{sizer_name}: Final Value = ${cerebro.broker.getvalue():,.2f}")

## 7. Analyzers

Analyzers compute performance metrics without affecting the strategy execution.

### Key Analyzers

In [ ]:
def run_with_analyzers(strategy_class, data_df, cash=100000, commission=0.001, **strategy_kwargs):
    """Helper function to run a strategy with full analyzer suite."""
    cerebro = bt.Cerebro()
    cerebro.adddata(bt.feeds.PandasData(dataname=data_df))
    cerebro.addstrategy(strategy_class, **strategy_kwargs)
    cerebro.broker.setcash(cash)
    cerebro.broker.setcommission(commission=commission)
    cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
    
    # Add analyzers
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', 
                        riskfreerate=0.0, timeframe=bt.TimeFrame.Days)
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    cerebro.addanalyzer(bt.analyzers.SQN, _name='sqn')
    
    results = cerebro.run()
    strat = results[0]
    
    # Print results
    sharpe = strat.analyzers.sharpe.get_analysis()
    dd = strat.analyzers.drawdown.get_analysis()
    ta = strat.analyzers.trades.get_analysis()
    ret = strat.analyzers.returns.get_analysis()
    sqn = strat.analyzers.sqn.get_analysis()
    
    print(f"{'='*50}")
    print(f"PERFORMANCE REPORT")
    print(f"{'='*50}")
    print(f"Total Return:    {ret.get('rtot', 0):.2%}")
    print(f"Ann. Return:     {ret.get('rnorm', 0):.2%}")
    sr = sharpe.get('sharperatio', None)
    print(f"Sharpe Ratio:    {sr:.4f}" if sr else "Sharpe Ratio:    N/A")
    print(f"Max Drawdown:    {dd.max.drawdown:.2f}%")
    print(f"Max DD Duration: {dd.max.len} bars")
    sqn_val = sqn.get('sqn', None)
    print(f"SQN:             {sqn_val:.2f}" if sqn_val else "SQN:             N/A")
    
    total = ta.get('total', {}).get('total', 0)
    won = ta.get('won', {}).get('total', 0)
    lost = ta.get('lost', {}).get('total', 0)
    print(f"\n--- Trades ---")
    print(f"Total:    {total}")
    print(f"Won:      {won}")
    print(f"Lost:     {lost}")
    if total > 0:
        print(f"Win Rate: {won/total:.1%}")
    
    return cerebro, results

# Run RSI strategy with full analysis
cerebro, results = run_with_analyzers(RSIStrategy, spy_data, printlog=False)

## 8. Complete Example: SMA Crossover with Full Analysis

Let us put everything together in a comprehensive example.

In [ ]:
class CompleteSMACrossover(bt.Strategy):
    """
    Complete SMA Crossover strategy with:
    - Configurable parameters
    - Order tracking
    - Trade logging
    """
    params = dict(
        fast_period=20,
        slow_period=50,
    )
    
    def __init__(self):
        self.sma_fast = bt.ind.SMA(self.data.close, period=self.p.fast_period)
        self.sma_slow = bt.ind.SMA(self.data.close, period=self.p.slow_period)
        self.crossover = bt.ind.CrossOver(self.sma_fast, self.sma_slow)
        
        self.order = None
        self.trades = []
        self.entry_size = 0
    
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        if order.status == order.Completed and order.isbuy():
            self.entry_size = order.executed.size
        self.order = None
    
    def notify_trade(self, trade):
        if trade.isclosed:
            entry_value = trade.price * abs(self.entry_size)
            self.trades.append({
                'entry': bt.num2date(trade.dtopen).date(),
                'exit': bt.num2date(trade.dtclose).date(),
                'bars': trade.barlen,
                'pnl': trade.pnl,
                'pnl_net': trade.pnlcomm,
                'return_pct': (trade.pnlcomm / entry_value * 100) if entry_value != 0 else 0.0,
            })
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()
    
    def stop(self):
        if self.trades:
            df = pd.DataFrame(self.trades)
            print(f"\n{'='*70}")
            print(f"TRADE LOG: SMA({self.p.fast_period}/{self.p.slow_period}) Crossover")
            print(f"{'='*70}")
            print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
            print(f"\nSummary: {len(df)} trades, "
                  f"Win Rate: {(df['pnl']>0).mean():.1%}, "
                  f"Avg Return: {df['return_pct'].mean():.2f}%, "
                  f"Avg Duration: {df['bars'].mean():.0f} bars")

# Run the complete example
cerebro, results = run_with_analyzers(
    CompleteSMACrossover, spy_data, 
    fast_period=20, slow_period=50, 
    cash=100000, commission=0.001
)

In [ ]:
# Plot the strategy
cerebro.plot(iplot=False,style='candle', volume=False, figsize=(16, 10))

## 9. Exercises

1. **RSI + SMA Strategy**: Create a backtrader strategy that:
   - Buys when RSI(14) < 30 AND price is above SMA(200)
   - Sells when RSI(14) > 70 OR price falls below SMA(200)
   - Test on SPY from 2015-2024 with 0.1% commission

2. **Custom Indicator**: Create a custom backtrader indicator that computes the z-score of price relative to a rolling mean:
   $$z_t = \frac{P_t - \text{SMA}(n)}{\sigma(n)}$$
   Use it in a mean-reversion strategy: buy when z < -2, sell when z > 0.

3. **Analyzer Comparison**: Run the same SMA crossover strategy with three different parameter sets: (10, 30), (20, 50), (50, 200). Use the `run_with_analyzers` function to compare performance. Which set has the best Sharpe ratio?

4. **Sizer Impact**: Compare the results of the MACD strategy using:
   - FixedSize(100 shares)
   - PercentSizer(50%)
   - PercentSizer(95%)
   How does position sizing affect returns and drawdown?

---
### References
- [Backtrader Indicators Reference](https://www.backtrader.com/docu/indautoref/)
- [Backtrader Order Types](https://www.backtrader.com/docu/order-creation-execution/order-creation-execution/)
- [Backtrader Analyzers](https://www.backtrader.com/docu/analyzers/analyzers/)
- John Murphy, *Technical Analysis of the Financial Markets*